# Análise Exploratória — Silver

Este notebook valida os dados da camada Silver e responde perguntas direcionadas
às queries do case antes de construir o `etl_gold.py`.

**Queries do case:**
1. Qual a média de `total_amount` recebido em um mês considerando todos os yellow táxis?
2. Qual a média de `passenger_count` por hora do dia em maio considerando todos os táxis?

**Database:** `ifood_case_silver`  
**Tabelas:** `table_yellow_taxi_silver` e `table_green_taxi_silver`

## 1. Importações e configurações

In [1]:
import logging
import warnings
import awswrangler as wr
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning, module="awswrangler")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] exploratory_silver - %(message)s",
    datefmt="%d-%m-%Y %H:%M:%S",
)
logger = logging.getLogger("exploratory_silver")

logging.getLogger("boto3").setLevel(logging.WARNING)
logging.getLogger("botocore").setLevel(logging.WARNING)
logging.getLogger("awswrangler").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

DATABASE  = "ifood_case_silver"
TABELAS   = ["table_yellow_taxi_silver", "table_green_taxi_silver"]
S3_OUTPUT = "s3://ifood-case-data-lake-kaique/athena-results/"

logger.info("Bibliotecas carregadas com sucesso")

08-06-2026 06:52:42 [INFO] exploratory_silver - Bibliotecas carregadas com sucesso


## 2. Verificação de disponibilidade das tabelas

Verificamos se as tabelas Silver foram criadas corretamente no Glue Catalog.

In [2]:
logger.info("Verificando tabelas no Glue Catalog | database=%s", DATABASE)

tabelas_catalog = wr.catalog.get_tables(database=DATABASE)
df_catalog = pd.DataFrame(tabelas_catalog)[["Name", "DatabaseName", "TableType"]]

logger.info("Tabelas encontradas | total=%s", len(df_catalog))
display(
    df_catalog.style
    .set_caption("Tabelas no Glue Catalog — ifood_case_silver")
    .set_properties(**{"text-align": "left"})
)

08-06-2026 06:52:45 [INFO] exploratory_silver - Verificando tabelas no Glue Catalog | database=ifood_case_silver
08-06-2026 06:52:46 [INFO] exploratory_silver - Tabelas encontradas | total=2


,Name,DatabaseName,TableType
0,table_green_taxi_silver,ifood_case_silver,EXTERNAL_TABLE
1,table_yellow_taxi_silver,ifood_case_silver,EXTERNAL_TABLE


## 3. Volume de dados por tabela e mês

Comparamos o volume do Silver com o Bronze para validar
que apenas os registros com nulos foram removidos.

In [3]:
logger.info("Iniciando analise de volume por tabela e mes")

volumes = []
for tabela in TABELAS:
    for mes in range(1, 6):
        df = wr.athena.read_sql_query(
            sql=f"""
                SELECT COUNT(*) as total_linhas
                FROM {tabela}
                WHERE partition_year = 2023
                AND   partition_month = {mes}
            """,
            database=DATABASE,
            s3_output=S3_OUTPUT,
        )
        total = df["total_linhas"].iloc[0]
        volumes.append({"tabela": tabela, "mes": f"2023-{str(mes).zfill(2)}", "linhas": total})
        logger.info("Volume | tabela=%s | mes=%s | linhas=%s", tabela, mes, f"{total:,}")

df_volumes = pd.DataFrame(volumes)
logger.info("Analise de volume concluida")

display(
    df_volumes
    .set_index(["tabela", "mes"])
    .rename(columns={"linhas": "Linhas"})
    .style
    .set_caption("Volume por Tabela e Mes — Silver")
    .format("{:,}", subset=["Linhas"])
    .set_properties(**{"text-align": "right"})
)

08-06-2026 06:52:51 [INFO] exploratory_silver - Iniciando analise de volume por tabela e mes
08-06-2026 06:53:01 [INFO] exploratory_silver - Volume | tabela=table_yellow_taxi_silver | mes=1 | linhas=2,995,023
08-06-2026 06:53:12 [INFO] exploratory_silver - Volume | tabela=table_yellow_taxi_silver | mes=2 | linhas=2,837,138
08-06-2026 06:53:23 [INFO] exploratory_silver - Volume | tabela=table_yellow_taxi_silver | mes=3 | linhas=3,316,147
08-06-2026 06:53:33 [INFO] exploratory_silver - Volume | tabela=table_yellow_taxi_silver | mes=4 | linhas=3,197,560
08-06-2026 06:53:43 [INFO] exploratory_silver - Volume | tabela=table_yellow_taxi_silver | mes=5 | linhas=3,411,853
08-06-2026 06:53:53 [INFO] exploratory_silver - Volume | tabela=table_green_taxi_silver | mes=1 | linhas=63,887
08-06-2026 06:54:04 [INFO] exploratory_silver - Volume | tabela=table_green_taxi_silver | mes=2 | linhas=59,988
08-06-2026 06:54:14 [INFO] exploratory_silver - Volume | tabela=table_green_taxi_silver | mes=3 | linha

## 4. Validação de nulos nas colunas obrigatórias

Verificamos se o Silver removeu todos os nulos nas colunas obrigatórias.
O Bronze tinha nulos apenas em `passenger_count` — esperamos zero nulos no Silver.

In [4]:
logger.info("Validando nulos nas colunas obrigatorias")

COLUNAS = ["vendor_id", "passenger_count", "total_amount", "pickup_datetime", "dropoff_datetime"]

for tabela in TABELAS:
    rows = []
    for col in COLUNAS:
        df = wr.athena.read_sql_query(
            sql=f"""
                SELECT
                    COUNT(*) as total,
                    COUNT({col}) as nao_nulos,
                    COUNT(*) - COUNT({col}) as nulos
                FROM {tabela}
            """,
            database=DATABASE,
            s3_output=S3_OUTPUT,
        )
        total  = df["total"].iloc[0]
        nulos  = df["nulos"].iloc[0]
        pct    = round(nulos / total * 100, 2) if total > 0 else 0
        rows.append({"coluna": col, "total": total, "nulos": nulos, "pct_nulos": pct})
        logger.info("Nulos | tabela=%s | coluna=%s | nulos=%s", tabela, col, f"{nulos:,}")

    df_nulos = pd.DataFrame(rows)
    display(
        df_nulos.set_index("coluna").style
        .set_caption(f"Nulos — {tabela}")
        .format("{:,}", subset=["total", "nulos"])
        .format("{:.2f}%", subset=["pct_nulos"])
        .bar(subset=["pct_nulos"], color="#ffcccc", vmin=0, vmax=100)
        .set_properties(**{"text-align": "left"})
    )

08-06-2026 06:54:34 [INFO] exploratory_silver - Validando nulos nas colunas obrigatorias
08-06-2026 06:54:44 [INFO] exploratory_silver - Nulos | tabela=table_yellow_taxi_silver | coluna=vendor_id | nulos=0
08-06-2026 06:54:55 [INFO] exploratory_silver - Nulos | tabela=table_yellow_taxi_silver | coluna=passenger_count | nulos=0
08-06-2026 06:55:05 [INFO] exploratory_silver - Nulos | tabela=table_yellow_taxi_silver | coluna=total_amount | nulos=0
08-06-2026 06:55:15 [INFO] exploratory_silver - Nulos | tabela=table_yellow_taxi_silver | coluna=pickup_datetime | nulos=0
08-06-2026 06:55:25 [INFO] exploratory_silver - Nulos | tabela=table_yellow_taxi_silver | coluna=dropoff_datetime | nulos=0


,total,nulos,pct_nulos
coluna,,,
vendor_id,"15,757,721",0,0.00%
passenger_count,"15,757,721",0,0.00%
total_amount,"15,757,721",0,0.00%
pickup_datetime,"15,757,721",0,0.00%
dropoff_datetime,"15,757,721",0,0.00%


08-06-2026 06:55:34 [INFO] exploratory_silver - Nulos | tabela=table_green_taxi_silver | coluna=vendor_id | nulos=0
08-06-2026 06:55:44 [INFO] exploratory_silver - Nulos | tabela=table_green_taxi_silver | coluna=passenger_count | nulos=0
08-06-2026 06:55:54 [INFO] exploratory_silver - Nulos | tabela=table_green_taxi_silver | coluna=total_amount | nulos=0
08-06-2026 06:56:04 [INFO] exploratory_silver - Nulos | tabela=table_green_taxi_silver | coluna=pickup_datetime | nulos=0
08-06-2026 06:56:14 [INFO] exploratory_silver - Nulos | tabela=table_green_taxi_silver | coluna=dropoff_datetime | nulos=0


,total,nulos,pct_nulos
coluna,,,
vendor_id,"316,734",0,0.00%
passenger_count,"316,734",0,0.00%
total_amount,"316,734",0,0.00%
pickup_datetime,"316,734",0,0.00%
dropoff_datetime,"316,734",0,0.00%


## 5. Análise de passenger_count — direcionada à Query 2

A Query 2 pede a média de passageiros por hora. Antes de agregar no Gold,
analisamos a distribuição do `passenger_count` para entender os dados.

Um táxi NYC comporta no máximo 4 passageiros (ou 5 com assento dianteiro).
Valores acima de 6 são considerados inválidos pela comunidade NYC Taxi.

In [5]:
logger.info("Analisando distribuicao de passenger_count")

for tabela in TABELAS:
    df = wr.athena.read_sql_query(
        sql=f"""
            SELECT
                passenger_count,
                COUNT(*) as total,
                ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct
            FROM {tabela}
            GROUP BY passenger_count
            ORDER BY passenger_count
        """,
        database=DATABASE,
        s3_output=S3_OUTPUT,
    )
    logger.info("Distribuicao carregada | tabela=%s", tabela)
    display(
        df.style
        .set_caption(f"Distribuicao de passenger_count — {tabela}")
        .format("{:,}", subset=["total"])
        .format("{:.2f}%", subset=["pct"])
        .bar(subset=["pct"], color="#cfe2ff", vmin=0, vmax=100)
        .set_properties(**{"text-align": "right"})
    )

08-06-2026 06:56:14 [INFO] exploratory_silver - Analisando distribuicao de passenger_count
08-06-2026 06:56:23 [INFO] exploratory_silver - Distribuicao carregada | tabela=table_yellow_taxi_silver


,passenger_count,total,pct
0,0,"273,481",1.74%
1,1,"11,894,120",75.48%
2,2,"2,356,679",14.96%
3,3,"577,606",3.67%
4,4,"300,125",1.90%
5,5,"217,068",1.38%
6,6,"138,530",0.88%
7,7,28,0.00%
8,8,65,0.00%
9,9,19,0.00%


08-06-2026 06:56:31 [INFO] exploratory_silver - Distribuicao carregada | tabela=table_green_taxi_silver


,passenger_count,total,pct
0,0,"2,407",0.76%
1,1,"269,630",85.13%
2,2,"26,383",8.33%
3,3,"4,528",1.43%
4,4,"1,441",0.45%
5,5,"7,908",2.50%
6,6,"4,407",1.39%
7,7,16,0.01%
8,8,8,0.00%
9,9,6,0.00%


## 6. Análise de total_amount — direcionada à Query 1

A Query 1 pede a média de `total_amount` por mês do Yellow Taxi.
Analisamos a distribuição para identificar outliers que possam distorcer a média.

In [6]:
logger.info("Analisando distribuicao de total_amount por mes")

df = wr.athena.read_sql_query(
    sql="""
        SELECT
            partition_month as mes,
            COUNT(*)                        as total_corridas,
            ROUND(AVG(total_amount), 2)     as media,
            ROUND(MIN(total_amount), 2)     as minimo,
            ROUND(MAX(total_amount), 2)     as maximo,
            ROUND(STDDEV(total_amount), 2)  as desvio_padrao,
            SUM(CASE WHEN total_amount <= 0 THEN 1 ELSE 0 END) as negativos_ou_zero
        FROM table_yellow_taxi_silver
        GROUP BY partition_month
        ORDER BY partition_month
    """,
    database=DATABASE,
    s3_output=S3_OUTPUT,
)

logger.info("Analise de total_amount concluida")
display(
    df.set_index("mes").style
    .set_caption("Distribuicao de total_amount por Mes — Yellow Taxi Silver")
    .format("{:,}", subset=["total_corridas", "negativos_ou_zero"])
    .format("{:.2f}", subset=["media", "minimo", "maximo", "desvio_padrao"])
    .set_properties(**{"text-align": "right"})
)

08-06-2026 06:56:31 [INFO] exploratory_silver - Analisando distribuicao de total_amount por mes
08-06-2026 06:56:42 [INFO] exploratory_silver - Analise de total_amount concluida


,total_corridas,media,minimo,maximo,desvio_padrao,negativos_ou_zero
mes,,,,,,
1,"2,995,023",26.97,-751.00,1169.40,22.27,"25,719"
2,"2,837,138",26.87,-757.55,2208.10,21.97,"25,401"
3,"3,316,147",27.76,-982.95,2100.00,22.98,"30,310"
4,"3,197,560",28.21,-807.55,2451.00,23.63,"30,263"
5,"3,411,853",28.87,-900.50,6304.90,24.07,"32,267"


## 7. Amostra dos dados

Verificamos visualmente os dados transformados no Silver.

In [7]:
logger.info("Carregando amostra dos dados")

for tabela in TABELAS:
    df = wr.athena.read_sql_query(
        sql=f"""
            SELECT *
            FROM {tabela}
            WHERE partition_year = 2023
            AND   partition_month = 1
            LIMIT 5
        """,
        database=DATABASE,
        s3_output=S3_OUTPUT,
    )
    logger.info("Amostra carregada | tabela=%s", tabela)
    display(
        df.style
        .set_caption(f"Amostra — {tabela} (Janeiro 2023)")
        .set_properties(**{"text-align": "left"})
    )

08-06-2026 06:56:42 [INFO] exploratory_silver - Carregando amostra dos dados
08-06-2026 06:56:52 [INFO] exploratory_silver - Amostra carregada | tabela=table_yellow_taxi_silver


,vendor_id,passenger_count,total_amount,pickup_datetime,dropoff_datetime,taxi_type,partition_year,partition_month
0,2,1,14.300000,2023-01-01 00:32:10,2023-01-01 00:40:36,yellow,2023,1
1,2,1,16.900000,2023-01-01 00:55:08,2023-01-01 01:01:27,yellow,2023,1
2,2,1,34.900000,2023-01-01 00:25:04,2023-01-01 00:37:49,yellow,2023,1
3,1,0,20.850000,2023-01-01 00:03:48,2023-01-01 00:13:25,yellow,2023,1
4,2,1,19.680000,2023-01-01 00:10:29,2023-01-01 00:21:19,yellow,2023,1


08-06-2026 06:57:03 [INFO] exploratory_silver - Amostra carregada | tabela=table_green_taxi_silver


,vendor_id,passenger_count,total_amount,pickup_datetime,dropoff_datetime,taxi_type,partition_year,partition_month
0,2,1,29.650000,2023-01-06 11:07:35,2023-01-06 11:36:16,green,2023,1
1,2,1,8.700000,2023-01-12 14:40:45,2023-01-12 14:47:09,green,2023,1
2,2,1,33.150000,2023-01-19 06:58:54,2023-01-19 07:16:59,green,2023,1
3,2,1,29.950000,2023-01-28 19:01:19,2023-01-28 19:15:31,green,2023,1
4,2,1,5.900000,2023-01-22 12:52:12,2023-01-22 12:53:26,green,2023,1


## 8. Conclusões

Com base na análise exploratória da camada Silver, as seguintes decisões foram tomadas para o `etl_gold.py`:

| Observação | Decisão no Gold |
|---|---|
| Nulos removidos com sucesso no Silver | Gold recebe dados limpos |
| `passenger_count` com valores > 6 existem | Filtrar `passenger_count` entre 1 e 6 no Gold para Query 2 |
| `total_amount` com valores negativos ainda presentes | Filtrar `total_amount > 0` no Gold para Query 1 |
| Yellow e Green com schema idêntico no Silver | UNION ALL viável no Gold |
| `taxi_type` presente no Silver | Filtrar `taxi_type = yellow` na Query 1 no Gold |